# HKI + LangChain: Domain-Isolated RAG

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/h3nok/HKI/blob/main/notebooks/02_langchain_rag.ipynb)

This notebook shows how to add HKI to an existing LangChain RAG pipeline.
The integration has two components:

- **`HkiCallbackHandler`** — wraps any chain; enforces the signed envelope on every LLM call, tool call, and retriever event
- **`HkiRetriever`** — wraps any LangChain retriever; drops documents that do not belong to the active domain before they reach the LLM

**Time to complete:** ~10 minutes  
**Requirements:** Python 3.11+, LangChain Core

In [ ]:
%pip install hki-runtime hki-langchain langchain-core -q

---
## Part 1 — The problem without HKI

A typical LangChain retriever returns every document it finds.
In a multi-domain system, a poorly configured retriever can return documents from the wrong domain.

In [ ]:
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from typing import List

# Simulate a knowledge base with documents from two domains
KNOWLEDGE_BASE = [
    Document(page_content="Pharmacy return policy: 30 days with receipt", metadata={"domain": "pharmacy", "id": "ph_001"}),
    Document(page_content="Hotel cancellation policy: 48 hours notice",   metadata={"domain": "travel",   "id": "tr_001"}),
    Document(page_content="Prescription pickup hours: Mon-Fri 9am-6pm",   metadata={"domain": "pharmacy", "id": "ph_002"}),
    Document(page_content="Flight rebooking fee: $150 per segment",        metadata={"domain": "travel",   "id": "tr_002"}),
]

class NaiveRetriever(BaseRetriever):
    """Returns ALL documents — no domain filtering."""
    def _get_relevant_documents(self, query: str, *, run_manager: CallbackManagerForRetrieverRun) -> List[Document]:
        return KNOWLEDGE_BASE  # BUG: returns everything regardless of domain

naive = NaiveRetriever()
docs = naive.invoke("what is the return policy")

print("Pharmacy agent received these documents:")
for d in docs:
    flag = "✓" if d.metadata["domain"] == "pharmacy" else "✗ LEAK"
    print(f"  {flag}  [{d.metadata['domain']}] {d.page_content}")

The pharmacy agent received travel documents. If those were drug pricing or member health records, this is a compliance violation.

---
## Part 2 — Add HkiRetriever to filter by active domain

In [ ]:
from hki_langchain import HkiRetriever

# A valid HKI envelope scoped to the pharmacy domain
PHARMACY_ENVELOPE = {
    "hki_version": "1.0",
    "envelope_id": "env_pharmacy_001",
    "org_id": "org_acme",
    "subject_id": "user_42",
    "active_domain": "pharmacy",
    "authorized_domains": ["pharmacy"],
    "purpose": "retrieve",
    "risk_tier": "read-only",
    "policy_pack_id": "pharmacy@2026-05",
    "issued_at": 0,
    "expires_at": 9_999_999_999,
    "issuer": "gateway.acme.internal",
    "signature": "ed25519:placeholder",
}

# Wrap the naive retriever with HKI — it will drop out-of-domain documents
safe_retriever = HkiRetriever(retriever=naive)

docs = safe_retriever.invoke(
    "what is the return policy",
    config={"metadata": {"hki_envelope": PHARMACY_ENVELOPE}},
)

print("Pharmacy agent received these documents (after HKI filtering):")
for d in docs:
    print(f"  ✓  [{d.metadata['domain']}] {d.page_content}")

print(f"\nTravel documents filtered out: {4 - len(docs)}")

---
## Part 3 — Add HkiCallbackHandler to a full RAG chain

The callback handler enforces the envelope on every event in the chain: LLM calls, tool calls, and retriever events. Any call that does not carry the envelope — or carries the wrong domain — is blocked.

In [ ]:
from hki_langchain import HkiCallbackHandler, HkiRetriever
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate

# Stub LLM — replace with your real LLM (OpenAI, Anthropic, Vertex, etc.)
def stub_llm(messages) -> str:
    context = messages[-1].content if hasattr(messages[-1], 'content') else str(messages[-1])
    return f"Based on the policy documents: {context[:80]}..."

# Build a simple RAG chain
prompt = ChatPromptTemplate.from_template(
    "Answer based only on these documents: {context}\n\nQuestion: {question}"
)

hki_retriever = HkiRetriever(retriever=naive)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

# The chain — HKI is wired in via config, not by changing the chain itself
rag_chain = (
    {"context": hki_retriever | RunnableLambda(format_docs), "question": RunnablePassthrough()}
    | prompt
    | RunnableLambda(stub_llm)
)

# Run the chain with the HKI callback handler
answer = rag_chain.invoke(
    "what is the return policy",
    config={
        "callbacks": [HkiCallbackHandler()],
        "metadata": {"hki_envelope": PHARMACY_ENVELOPE},
    },
)

print(answer)

---
## Part 4 — What happens when the envelope is missing or wrong domain

In [ ]:
# Case 1: No envelope at all
try:
    safe_retriever.invoke("what is the return policy")  # no config passed
    print("ERROR: should have been blocked")
except Exception as e:
    print(f"No envelope → blocked: {e}")

print()

# Case 2: Wrong domain — travel envelope trying to read pharmacy docs
TRAVEL_ENVELOPE = {**PHARMACY_ENVELOPE, "active_domain": "travel", "authorized_domains": ["travel"]}

docs = safe_retriever.invoke(
    "what is the return policy",
    config={"metadata": {"hki_envelope": TRAVEL_ENVELOPE}},
)

print(f"Travel envelope, pharmacy docs → {len(docs)} documents returned")
if docs:
    for d in docs:
        print(f"  [{d.metadata['domain']}] {d.page_content}")
else:
    print("  (no pharmacy documents visible to travel domain — correct)")

---
## Part 5 — Domain-bound cache keys

HKI prevents cache collisions between domains. Use `hki_cache_key` with any LangChain `BaseCache`.

In [ ]:
from hki_langchain import hki_cache_key

pharmacy_key = hki_cache_key(PHARMACY_ENVELOPE, "what is the return policy")
travel_key   = hki_cache_key(TRAVEL_ENVELOPE,   "what is the return policy")

print(f"pharmacy key: {pharmacy_key}")
print(f"travel key:   {travel_key}")
print(f"Are they different? {pharmacy_key != travel_key}")

---
## Summary — what changed in your LangChain code

| Without HKI | With HKI |
|---|---|
| `retriever.invoke(query)` | `HkiRetriever(retriever=r).invoke(query, config={...})` |
| `chain.invoke(input)` | `chain.invoke(input, config={"callbacks": [HkiCallbackHandler()], "metadata": {"hki_envelope": env}})` |
| `cache[query] = result` | `cache[hki_cache_key(env, query)] = result` |

Three lines of change. The chain itself is untouched — HKI wraps the boundary, not the logic.

### Next steps
- [03 — FastAPI middleware](./03_fastapi_middleware.ipynb): enforce HKI at the HTTP layer
- [04 — Threat demos](./04_threat_demos.ipynb): all 15 threats, before and after
- [Conformance guide](../docs/HKI_CONFORMANCE.md): test your implementation